This notebook focusses on taking structured logs and turn them into actionable data that a detection engineer (or an ML model later) can use.


In [53]:
import sqlite3

In [54]:
conn = sqlite3.connect('../../01_data_pipeline/security_logs.db')
cursor = conn.cursor()

In [55]:
conn.close()
conn = sqlite3.connect('../../01_data_pipeline/security_logs.db')
cursor = conn.cursor()

In [56]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS events (
    event_id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp DATETIME,
    host TEXT,
    process_name TEXT,
    pid INTEGER,
    message TEXT
)
''')

conn.commit()

In [57]:
import pandas as pd

df = pd.read_csv('../datasets/processed/system_events.csv')
for _, row in df.iterrows():
    cursor.execute('''
    INSERT INTO events (timestamp, host, process_name, pid, message)
    VALUES (?, ?, ?, ?, ?)
    ''', (row['timestamp'], row['host'], row['process_name'], row['pid'], row['message']))

conn.commit()

In [58]:
# convert timestamp to true date time and remove rows with invalid timestamps
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp'])

In [ ]:
# determine hour of day, day of week, and whether it's a weekend
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = df['day_of_week'] >= 5

In [60]:
# macOS aware off-hours detection: 8pm to 6am on weekdays, all day weekends
df['off_hours'] = df.apply(
    lambda x: 1 if (x['hour'] < 6 or x['hour'] > 20 or x['is_weekend']) else 0,
    axis=1
)

In [61]:
# flag system processes to ignore in anomaly detection
system_processes = ['kernel_task', 'kernel', 'launchd', 'syslogd', 'systemd', 'init', 'UserEventAgent']

df['is_system_process'] = df['process_name'].isin(system_processes).astype(int)

In [62]:
# rare processes detection
user_df = df[df['is_system_process'] == 0]

process_counts = user_df['process_name'].value_counts()

if len(process_counts) > 0:
    rare_threshold = process_counts.quantile(0.05)
    rare_processes = process_counts[process_counts <= rare_threshold].index
else:
    rare_processes = []

df['rare_process'] = df['process_name'].isin(rare_processes).astype(int)

In [63]:
# first seen process time detection
first_seen = df.groupby('process_name')['timestamp'].min().reset_index()
first_seen.columns = ['process_name', 'first_seen_time']

df = df.merge(first_seen, on='process_name', how='left')

df['first_seen'] = (df['timestamp'] == df['first_seen_time']).astype(int)

In [64]:
# session awareness detection (sleep/wake)
df = df.sort_values('timestamp')

df['session_id'] = (df['timestamp'].diff() > pd.Timedelta('1h')).cumsum()
df['first_seen_session'] = (
    df.groupby(['process_name', 'session_id']).cumcount() == 0
).astype(int)

In [ ]:
# burst detection: count of events in the last hour
df = df.sort_values(['process_name', 'timestamp']).copy()

# Create temporary count column
df['_temp'] = 1

# Rolling sum with proper grouping
result = (
    df.set_index('timestamp')
    .groupby('process_name', group_keys=False)['_temp']
    .rolling('1h')
    .sum()
)

df['count_last_hour'] = result.values
df = df.drop('_temp', axis=1)
df['count_last_hour'] = df['count_last_hour'].fillna(1)

In [68]:
df.to_csv('../datasets/enriched/system_events_enriched.csv', index=False)